# 00 — Environment check (Colab Pro, T4 reference GPU)

Phase 0 of the Financial Reasoning SLM pipeline. Verifies the Colab GPU runtime, installs the open stack,
confirms every library we depend on imports cleanly, and runs a 4-bit smoke test of Llama-3.2-3B-Instruct.

**Kernel:** Colab server with a **T4 GPU** runtime (the reference hardware for all resume-claimed training metrics; L4/A100 are used only for eval and generation jobs). Output: `environment.json` (printed at the end so it is saved in this notebook's outputs).

The smoke-test numbers are a sanity check only; they are **not** resume metrics.

In [1]:
# Install the open stack. Unsloth pulls in compatible torch / transformers / trl / peft / bitsandbytes.
%pip install -q unsloth
%pip install -q sentence-transformers faiss-cpu datasets evaluate rouge-score psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 85.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
import unsloth  # must be imported before transformers/trl/peft so its patches apply
import platform, shutil, subprocess, sys
import psutil, torch

assert torch.cuda.is_available(), "No GPU — switch the Colab runtime to T4 GPU and reconnect."
props = torch.cuda.get_device_properties(0)
gpu = {
    "name": props.name,
    "vram_gb": round(props.total_memory / 1024**3, 2),
    "compute_capability": f"{props.major}.{props.minor}",
    "bf16_supported": torch.cuda.is_bf16_supported(),
}
host = {
    "python": platform.python_version(),
    "ram_gb": round(psutil.virtual_memory().total / 1024**3, 2),
    "cpu_count": psutil.cpu_count(),
    "disk_free_gb": round(shutil.disk_usage("/").free / 1024**3, 2),
}
print(gpu, host, sep="\n")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
{'name': 'Tesla T4', 'vram_gb': 14.56, 'compute_capability': '7.5', 'bf16_supported': False}
{'python': '3.13.15', 'ram_gb': 12.67, 'cpu_count': 2, 'disk_free_gb': 187.64}
Sun Sep 27 02:47:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T

In [3]:
# Library versions, plus where ORPOTrainer lives (recent TRL moved it under trl.experimental).
from importlib.metadata import version, PackageNotFoundError

packages = ["torch", "transformers", "trl", "peft", "unsloth", "bitsandbytes", "accelerate",
            "datasets", "sentence-transformers", "faiss-cpu", "evaluate"]
versions = {}
for p in packages:
    try:
        versions[p] = version(p)
    except PackageNotFoundError:
        versions[p] = None
versions["cuda"] = torch.version.cuda

orpo_import = None
for path in ("trl", "trl.experimental.orpo"):
    try:
        getattr(__import__(path, fromlist=["ORPOTrainer"]), "ORPOTrainer")
        orpo_import = path
        break
    except (ImportError, AttributeError):
        pass

from trl import SFTTrainer  # noqa: F401
import faiss, sentence_transformers  # noqa: F401

print(versions)
print("ORPOTrainer import path:", orpo_import)
assert orpo_import, "ORPOTrainer not found in this TRL version — flag before Phase 10."

{'torch': '2.11.0+cu128', 'transformers': '5.5.0', 'trl': '0.24.0', 'peft': '0.20.0', 'unsloth': '2026.9.11', 'bitsandbytes': '0.50.2', 'accelerate': '1.14.0', 'datasets': '4.3.0', 'sentence-transformers': '5.7.0', 'faiss-cpu': '1.15.1', 'evaluate': '0.4.6', 'cuda': '12.8'}
ORPOTrainer import path: trl


In [4]:
# 4-bit smoke test. Unsloth's mirror of Llama-3.2-3B-Instruct is ungated (Llama 3.2 Community License still applies).
import time
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
torch.cuda.reset_peak_memory_stats()
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_ID, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
load_peak_gb = torch.cuda.max_memory_allocated() / 1024**3

messages = [{"role": "user", "content": "Revenue was $100M in 2022 and $112.4M in 2023. What is the percentage change?"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

torch.cuda.synchronize(); t0 = time.perf_counter()
out = model.generate(input_ids=inputs, max_new_tokens=128, do_sample=False)
torch.cuda.synchronize(); elapsed = time.perf_counter() - t0

new_tokens = out.shape[1] - inputs.shape[1]
smoke = {
    "model": MODEL_ID,
    "load_peak_vram_gb": round(load_peak_gb, 2),
    "generate_peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "new_tokens": int(new_tokens),
    "hf_generate_tok_s": round(new_tokens / elapsed, 1),
}
print(tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True))
print(smoke)

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To find the percentage change, first, find the difference between the two years:

$112.4M - $100M = $12.4M

Then, divide the difference by the original amount and multiply by 100:

($12.4M / $100M) * 100 = 12.4%

So, the revenue increased by 12.4% in 2023.
{'model': 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit', 'load_peak_vram_gb': 2.15, 'generate_peak_vram_gb': 2.26, 'new_tokens': 83, 'hf_generate_tok_s': 12.0}


In [5]:
import json, datetime

environment = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "runtime": "google-colab-pro",
    "gpu": gpu,
    "host": host,
    "versions": versions,
    "orpo_import": orpo_import,
    "smoke_test": smoke,
}
with open("environment.json", "w") as f:
    json.dump(environment, f, indent=2)
print(json.dumps(environment, indent=2))

{
  "created_utc": "2026-09-27T02:48:23+00:00",
  "runtime": "google-colab-pro",
  "gpu": {
    "name": "Tesla T4",
    "vram_gb": 14.56,
    "compute_capability": "7.5",
    "bf16_supported": false
  },
  "host": {
    "python": "3.13.15",
    "ram_gb": 12.67,
    "cpu_count": 2,
    "disk_free_gb": 187.64
  },
  "versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.5.0",
    "trl": "0.24.0",
    "peft": "0.20.0",
    "unsloth": "2026.9.11",
    "bitsandbytes": "0.50.2",
    "accelerate": "1.14.0",
    "datasets": "4.3.0",
    "sentence-transformers": "5.7.0",
    "faiss-cpu": "1.15.1",
    "evaluate": "0.4.6",
    "cuda": "12.8"
  },
  "orpo_import": "trl",
  "smoke_test": {
    "model": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    "load_peak_vram_gb": 2.15,
    "generate_peak_vram_gb": 2.26,
    "new_tokens": 83,
    "hf_generate_tok_s": 12.0
  }
}
